In [ ]:
import tensorflow as tf
import os
import numpy as np
import keras
from keras import layers
from tensorflow import data as tf_data
import matplotlib.pyplot as plt

data_dir = "MushroomPics"
dataset = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    labels="inferred",
    label_mode="categorical",
    image_size=(224, 224),
    batch_size=16,
    shuffle=True
)
#dataset = dataset.apply(tf.data.experimental.ignore_errors())

#length = len(dataset)
length = tf.data.experimental.cardinality(dataset).numpy()

train_size = int(length * 0.7)
test_size = int(length * 0.15)

"""
train = dataset.take(train_size)
test = dataset.skip(train_size).take(test_size)
val = dataset.skip(train_size + test_size)
"""
train = dataset.take(train_size)
remaining = dataset.skip(train_size)
test = remaining.take(test_size)
val = remaining.skip(test_size)

print(dataset.class_names)
print(dataset)

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
])

In [ ]:
model = tf.keras.Sequential([
    layers.Rescaling(1./255, input_shape=(224, 224, 3)),
    data_augmentation,
    
    layers.Conv2D(32, (9,9), activation='relu'),
    layers.MaxPooling2D(),
    
    layers.Conv2D(64, (9,9), activation='relu'),
    layers.MaxPooling2D(),
    
    layers.Conv2D(128, (9,9), activation='relu'),
    layers.MaxPooling2D(),
    
    layers.Conv2D(256, (9,9), activation='relu'),
    layers.MaxPooling2D(),
    
    #layers.Flatten(),
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    
    layers.Dense(len(dataset.class_names))
])

In [ ]:
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train,
    validation_data=val,
    epochs=64
)

In [ ]:
import matplotlib.pyplot as plt

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

epochs = range(len(acc))

plt.plot(epochs, acc, label='Training accuracy')
plt.plot(epochs, val_acc, label='Validation accuracy')
plt.legend()
plt.title('Accuracy')
plt.show()

plt.plot(epochs, loss, label='Training loss')
plt.plot(epochs, val_loss, label='Validation loss')
plt.legend()
plt.title('Loss')
plt.show()

In [ ]:
test_loss, test_acc = model.evaluate(test)
val_loss, val_acc = model.evaluate(val)
print("Test accuracy:", test_acc)
print("Validation accuracy:", val_acc)

In [ ]:
import tensorflow as tf

for root, _, files in os.walk("Mushrooms"):
    for f in files:
        path = os.path.join(root, f)
        try:
            img = tf.io.read_file(path)
            tf.io.decode_image(img)
        except Exception as e:
            print("BAD FILE:", path)